In [1]:
# 基础数据处理库
import numpy as np
import pandas as pd

# 单细胞分析标准库
import anndata as ad
import scanpy as sc

# 处理超大型 HDF5 文件的底层库
import h5py

In [2]:
file_path = "../data/interim/Parse_10M_PBMC_PBS_ordered.h5ad" 

# 使用 backed 模式（只读模式）
print("正在以 backed 模式打开文件...")
adata_backed = ad.read_h5ad(file_path, backed='r')

# 打印整体概况
print(adata_backed)

正在以 backed 模式打开文件...
AnnData object with n_obs × n_vars = 629701 × 19295 backed at '../data/interim/Parse_10M_PBMC_PBS_ordered.h5ad'
    obs: 'sample', 'species', 'gene_count', 'tscp_count', 'mread_count', 'bc1_wind', 'bc2_wind', 'bc3_wind', 'bc1_well', 'bc2_well', 'bc3_well', 'log1p_n_genes_by_counts', 'log1p_total_counts', 'total_counts_MT', 'pct_counts_MT', 'log1p_total_counts_MT', 'donor', 'cytokine', 'treatment', 'cell_type'
    var: 'Symbol', 'PBS_Index'


In [3]:
# obs: 查看前 5 个细胞的元数据
display(adata_backed.obs.head(5))

,sample,species,gene_count,tscp_count,mread_count,bc1_wind,bc2_wind,bc3_wind,bc1_well,bc2_well,bc3_well,log1p_n_genes_by_counts,log1p_total_counts,total_counts_MT,pct_counts_MT,log1p_total_counts_MT,donor,cytokine,treatment,cell_type
91_103_005__s1,Donor10_PBS,hg38,1797,3122,5858,91,103,5,H7,p2.A7,A5,7.494430,8.046550,18.0,0.576553,2.944439,Donor10,PBS,PBS,CD4 Naive
91_103_059__s1,Donor10_PBS,hg38,2945,7927,14656,91,103,59,H7,p2.A7,E11,7.988204,8.978156,82.0,1.034439,4.418840,Donor10,PBS,PBS,CD4 Naive
91_115_049__s1,Donor10_PBS,hg38,2906,6564,12330,91,115,49,H7,p2.B7,E1,7.974877,8.789508,123.0,1.873857,4.820282,Donor10,PBS,PBS,CD14 Mono
91_115_074__s1,Donor10_PBS,hg38,2874,6344,11814,91,115,74,H7,p2.B7,G2,7.963808,8.755423,134.0,2.112232,4.905275,Donor10,PBS,PBS,CD8 Naive
91_115_107__s1,Donor10_PBS,hg38,2581,4933,9263,91,115,107,H7,p2.B7,p2.A11,7.856320,8.503905,59.0,1.196027,4.094345,Donor10,PBS,PBS,CD14 Mono


In [4]:
# 统计每个处理条件下的细胞数
display(adata_backed.obs['treatment'].value_counts())

treatment
PBS    629701
Name: count, dtype: int64

In [5]:
# 统计每个 cytokine 处理条件下的细胞数
display(adata_backed.obs['cytokine'].value_counts())

cytokine
PBS    629701
Name: count, dtype: int64

In [6]:
# 统计每个细胞类型下的细胞数
display(adata_backed.obs['cell_type'].value_counts())

cell_type
CD4 Memory               152889
CD14 Mono                110999
CD4 Naive                108139
CD8 Memory                47563
B Naive                   41445
CD8 Naive                 37922
NK                        33085
MAIT                      21539
B Intermediate/Memory     21350
CD16 Mono                 16710
NKT                        9977
Treg                       9883
NK CD56bright              7870
cDC                        7109
pDC                        1273
HSPC                       1035
ILC                         510
Plasmablast                 403
Name: count, dtype: int64

In [7]:
# var: 查看前 5 个基因的元数据
display(adata_backed.var.head(5))

,Symbol,PBS_Index
Index,,
0,A1BG,A1BG
1,A1CF,A1CF
2,A2M,A2M
3,A2ML1,A2ML1
4,A3GALT2,A3GALT2


In [15]:
# X: 查看前 5 个细胞和前 5 个基因的主表达矩阵
import scipy.sparse

# 安全地切片：只把前 5 个细胞和前 5 个基因的代码/指针切出来
matrix_view = adata_backed[:5, :5]

# 将这部分微型数据彻底加载到内存中
mini_adata = matrix_view.to_memory()

# 提取表达矩阵 X
X_chunk = mini_adata.X

# 单细胞数据中 X 通常是稀疏矩阵 (Sparse Matrix)，需要转换为稠密数组以便阅读
if scipy.sparse.issparse(X_chunk):
    X_chunk = X_chunk.toarray()

# 用 Pandas DataFrame 包装，并带上真实的细胞名 (obs_names) 和基因名 (var_names)
view_df = pd.DataFrame(
    X_chunk,
    index=mini_adata.obs_names,
    columns=mini_adata.var_names
)

# 打印出来
print("前 5 个细胞 × 前 5 个基因 的主矩阵内容：")
display(view_df)

前 5 个细胞 × 前 5 个基因 的主矩阵内容：


Index,0,1,2,3,4
91_103_005__s1,0.0,0.0,0.0,0.0,0.0
91_103_059__s1,0.0,0.0,0.0,0.0,0.0
91_115_049__s1,0.0,0.0,0.0,0.0,0.0
91_115_074__s1,0.0,0.0,0.0,0.0,0.0
91_115_107__s1,0.0,0.0,0.0,0.0,0.0


In [9]:
# obsm: 细胞的多维注释 (通常是 PCA, UMAP, t-SNE 坐标)
print("\n细胞降维坐标 (obsm):")
if adata_backed.obsm:
    print("包含的主键:", list(adata_backed.obsm.keys()))
else:
    print("[空]")


细胞降维坐标 (obsm):
[空]


In [10]:
# uns: 非结构化数据 (通常是工具运行参数、聚类颜色等杂项)
print("\n非结构化元数据 (uns):")
if adata_backed.uns:
    print("包含的主键:", list(adata_backed.uns.keys()))
else:
    print("[空]")


非结构化元数据 (uns):
[空]


In [11]:
# layers: 额外的数据层 (通常存放 'counts' 原始计数，或 'spliced'/'unspliced' 剪切数据)
print("\n额外数据层 (layers):")
if adata_backed.layers:
    print("包含的数据层:", list(adata_backed.layers.keys()))
else:
    print("[空]")


额外数据层 (layers):
[空]


In [12]:
# raw: 原始数据的完整备份 (在过滤或寻找高变基因前的数据)
print("\n原始数据备份 (raw):")
if adata_backed.raw is not None:
    print(f"[存在] 包含了 {adata_backed.raw.shape[1]} 个原始基因")
else:
    print("[无]")


原始数据备份 (raw):
[无]


In [13]:
# obsp / varp / varm: 图谱和基因多维数据 (通常是 KNN 邻接矩阵等)
print("\n其他高级图谱及矩阵 (obsp/varm/varp):")
print("varm (基因多维数据):", list(adata_backed.varm.keys()) if adata_backed.varm else "[空]")
print("obsp (细胞间图谱):", list(adata_backed.obsp.keys()) if adata_backed.obsp else "[空]")
print("varp (基因间图谱):", list(adata_backed.varp.keys()) if adata_backed.varp else "[空]")


其他高级图谱及矩阵 (obsp/varm/varp):
varm (基因多维数据): [空]
obsp (细胞间图谱): [空]
varp (基因间图谱): [空]
